In [151]:
# Sam Brown
# sam_brown@mines.edu
# July 22
# Goal: make model that can predict whether a high or low tide will occur next, this information is important to predict slip size and inter evt time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report, precision_score, f1_score
df = pd.read_csv("/Users/sambrown04/Documents/SURF/Preproc_data/10-18.csv")



In [123]:
# Will use the past 10 days (adjustable) to predict whether the next event is a high tide or low tide event
# Will also use the step plot as a feature.
sequence = [1 if x == 1 else -1 for x in df['high_t_evt']]

df['cumult'] = np.cumsum(sequence)

df = df.dropna().reset_index(drop=True)

feature_cols = ['high_t_evt', 'cumult']

seq = 10

X = []
y = []

# Loop over data set and make sets of seq
for i in range(0, len(df['high_t_evt']) - seq):
    features = df.loc[i:i+seq-1, feature_cols].values
    targets = df.loc[i + seq, 'high_t_evt']
    
    X.append(features)
    y.append(targets)
    
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)  # num samples, seq_length, num_features (for this it is just the sequence of high and low evts and the step plt vals
print("y shape:", y.shape)

X shape: (4565, 10, 2)
y shape: (4565,)


In [61]:
# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # (N, 1)

# Flatten 
X_flat = X_tensor.view(X_tensor.shape[0], -1)

# Dataloader
dataset = TensorDataset(X_flat, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [125]:
class RegNet(nn.Module):
    def __init__(self, input_dim):
        super(RegNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model_dense = RegNet(input_dim=X_flat.shape[1])

In [127]:
# dataset_seq = TensorDataset(X_tensor, y_tensor)
# loader_seq = DataLoader(dataset_seq, batch_size=32, shuffle=True)

# Prepare for training 
train_size = int(.8 * len(X_tensor))

X_train = X_tensor[:train_size]
y_train = y_tensor[:train_size]
X_test = X_tensor[train_size:]
y_test = y_tensor[train_size:]

train_loader_ls = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=False)
test_loader_ls  = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

In [129]:
class LSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)  # Only use final hidden state
        return self.fc(h_n.squeeze(0))  # shape: (batch, hidden_size)

model_lstm = LSTMNet(input_size=X_tensor.shape[2])

In [169]:
def train(model, loader, epochs=10):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        total_loss = 0
        for xb, yb in loader:
            pred = model(xb)
            loss = criterion(pred, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

In [171]:
train(model_dense, loader)
train(model_lstm, train_loader_ls)


Epoch 1: Loss = 1140.7351
Epoch 2: Loss = 129.5452
Epoch 3: Loss = 116.7076
Epoch 4: Loss = 111.3302
Epoch 5: Loss = 108.6558
Epoch 6: Loss = 104.7234
Epoch 7: Loss = 108.6279
Epoch 8: Loss = 102.8956
Epoch 9: Loss = 185.0135
Epoch 10: Loss = 98.4520
Epoch 1: Loss = 84.9778
Epoch 2: Loss = 69.8239
Epoch 3: Loss = 66.4081
Epoch 4: Loss = 66.6547
Epoch 5: Loss = 66.3294
Epoch 6: Loss = 66.3416
Epoch 7: Loss = 66.2480
Epoch 8: Loss = 66.3916
Epoch 9: Loss = 66.3822
Epoch 10: Loss = 65.9613


In [173]:
def test(model, loader):
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for bx, by in loader:
            pred = model(bx)  # shape (batch_size, 1)
            pred_labels = (pred > 0.5).int()
            preds.append(pred_labels.cpu().numpy())
            trues.append(by.cpu().numpy())

    preds = np.concatenate(preds).reshape(-1)
    trues = np.concatenate(trues).reshape(-1)

    accuracy = accuracy_score(trues, preds)
    precision = precision_score(trues, preds, average='binary', zero_division=0)
    recall = recall_score(trues, preds, average='binary', zero_division=0)
    f1 = f1_score(trues, preds, average='binary', zero_division=0)
    cm = confusion_matrix(trues, preds)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    

In [167]:
test(model_lstm, test_loader_ls)

Accuracy: 0.2563
Precision: 0.0000
Recall: 0.0000
F1-Score: 0.0000
Confusion Matrix:
 [[234   0]
 [679   0]]
